# 04 - curated h5ad を merge

**anndata ネイティブの `ad.concat`** で結合する（自作 merge ユーティリティは使わない）。遺伝子は `gene_symbol_upper` を var_names に揃え、`join='outer'`。正規化・batch correction・scVI はしない。

- **merge 1**: `raw_or_filtered_count` のみ
- **merge 2**: 全 curated（`data_status` を obs に残す status-aware）

In [ ]:
import sys
from pathlib import Path

# config/dataset_manifest.yaml を持つ v2 ルートを探して src を import パスに追加
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

import manifest_utils as mf
man = mf.load_manifest()
paths = mf.project_paths(ROOT)
mf.ensure_dirs(paths)

import anndata as ad
import pandas as pd
import anndata_utils as au

curated = au.load_h5ad_collection(paths['curated'])
{k: (a.n_obs, a.n_vars, str(a.obs['data_status'].iloc[0])) for k, a in curated.items()}

## gene_symbol_upper を var_names に揃える小関数

In [ ]:
def on_symbol_upper(a):
    b = a.copy()
    b.var_names = pd.Index(b.var['gene_symbol_upper'].astype(str))
    b.var_names_make_unique()
    return b

## merge 1 - raw_or_filtered_count のみ

In [ ]:
raw_keys = [k for k, a in curated.items()
            if (a.obs['data_status'].astype(str) == 'raw_or_filtered_count').all()]
print('included:', raw_keys)
merged_raw = ad.concat([on_symbol_upper(curated[k]) for k in raw_keys],
                       join='outer', merge='same', index_unique=None)
merged_raw.obs_names_make_unique()
au.save_h5ad(merged_raw, paths['merged'] / 'merged_raw_or_filtered_count_als_sma_mouse.h5ad')
merged_raw

## merge 2 - 全 curated（status-aware）

In [ ]:
merged_all = ad.concat([on_symbol_upper(a) for a in curated.values()],
                       join='outer', merge='same', index_unique=None)
merged_all.obs_names_make_unique()
print(merged_all.obs['data_status'].value_counts())
au.save_h5ad(merged_all, paths['merged'] / 'merged_all_status_aware_als_sma_mouse.h5ad')
merged_all

## merge 後の細胞数表（pandas で集計）

In [ ]:
for key in ['source_accession', 'dataset_id', 'disease_status', 'treatment',
            'tissue', 'enrichment', 'data_status']:
    if key in merged_all.obs.columns:
        print(f'=== {key} ===')
        print(merged_all.obs[key].astype(str).value_counts().to_string())
        print()

次: **05_check_merged_h5ad.ipynb**